In [1]:
# #set up paths
import importlib
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# #This block only important for running as script
# script_dir = os.path.dirname(os.path.abspath(__file__))
# mariposa_dir = os.path.dirname(script_dir)
# utils_dir = os.path.join(mariposa_dir, 'utils')
# sys.path.append(utils_dir)
# sys.path.append(mariposa_dir)

#import utils
from utils import analysis, plot, metadata

importlib.reload(analysis)
importlib.reload(plot)
importlib.reload(metadata)

<module 'utils.metadata' from '/Users/snewbank/PycharmProjects/MARIPOSA/utils/metadata.py'>

In [4]:
config_path="/Users/snewbank/Behavior/MARIPOSA_test/240827_BSOID-test/config_PS.yaml"
config = metadata.load_project(config_path)
save_path="/Users/snewbank/Behavior/MARIPOSA_test/240827_BSOID-test/"
save=False

In [ ]:
# Linear discriminant analysis - scan
binsizes=[0.5*60,1*60,2.5*60,5*60,10*60,20*60]
accuracies=[]
for binsize in binsizes:
    labels_df, n_modules = analysis.label_counter_subgroups(config,0,1200)
    lda, lda_embeddings, label_counts, group_labels, group_dict, nbins = analysis.lda_labels_timebins(config,labels_df,binsize)
    confusion, class_num, class_labels, accuracy = analysis.loocv_conf_mat(lda, label_counts, group_labels, group_dict)
    accuracies.append(accuracy)

loocv_df = pd.DataFrame({"binsize": binsizes, "LOOCV": accuracies})
best_bin_arg=np.argmax(loocv_df.LOOCV)
best_bin=loocv_df.binsize[best_bin_arg]
fig = plt.figure(figsize=(4,2.5))
plt.plot(loocv_df.binsize,loocv_df.LOOCV,color="black",marker="o")
plt.xlabel("Bin size (s)")
plt.ylabel("Linear Discriminant Analysis \nLOOCV Accuracy")
fig.tight_layout()
plt.show()
# if save==True:
#     plt.savefig(save_path+"lda_loocv_scan.png",dpi=500)
#     loocv_df.to_csv(save_path+"lda_loocv.csv")

# Linear discriminant analysis - plot
binsize=best_bin
labels_df, n_modules = analysis.label_counter_subgroups(config,0,1200)
lda, lda_embeddings, label_counts, group_labels, group_dict, nbins = analysis.lda_labels_timebins(config,labels_df,binsize)
fig = plot.plot_lda(config, lda, lda_embeddings, group_labels, nbins, binsize, cmap="viridis_r")
plt.savefig(save_path+"lda_embeddings_2p5m.png",dpi=500)
confusion, class_num, class_labels, accuracy = analysis.loocv_conf_mat(lda, label_counts, group_labels, group_dict)
plot.plot_conf_mat(confusion, class_num, class_labels,alt_title="Linear Discriminant Analysis\nConfusion Matrix")
plt.show()
# if save == True:
#     plt.savefig(save_path+"lda_confmat_2p5m.png",dpi=500)